# CloudTube - A YouTube Uploader

Upload videos to YouTube directly from **Google Drive** or a **direct URL** — no local download required.

The video streams through this Colab VM's memory/tmp storage and goes straight to YouTube.

## Prerequisites

- A Google account with a YouTube channel
- That's it! OAuth credentials are bundled with this notebook.

## 1. Install Dependencies

In [ ]:
!pip install -q google-auth google-auth-oauthlib google-auth-httplib2 google-api-python-client requests tqdm

## 2. Load OAuth Credentials

Credentials are bundled with this notebook. If you're running in Colab, they'll be fetched automatically from the repo.

In [ ]:
import os

CREDS_URL = "https://raw.githubusercontent.com/mathinraj/CloudTube/main/client_secret.json"

if not os.path.exists('client_secret.json'):
    print("Fetching bundled credentials from repo...")
    os.system(f'wget -q -O client_secret.json "{CREDS_URL}"')

if os.path.exists('client_secret.json') and os.path.getsize('client_secret.json') > 0:
    print("Credentials loaded.")
else:
    raise FileNotFoundError(
        "client_secret.json not found. Clone the full repo or download it from:\n"
        f"  {CREDS_URL}"
    )

## 3. Authenticate with Google

This will print an authorization URL. Open it in your browser, sign in, and grant access. After authorizing, you'll be redirected to a page that won't load — just copy the full URL from the address bar and paste it back here.

In [ ]:
import json
import os
import pickle
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
from googleapiclient.discovery import build

os.environ['OAUTHLIB_INSECURE_TRANSPORT'] = '1'

SCOPES = [
    'https://www.googleapis.com/auth/youtube.upload',
    'https://www.googleapis.com/auth/youtube',
    'https://www.googleapis.com/auth/drive.readonly',
]

TOKEN_FILE = 'token.pickle'

def authenticate():
    creds = None
    if os.path.exists(TOKEN_FILE):
        with open(TOKEN_FILE, 'rb') as f:
            creds = pickle.load(f)

    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(
                'client_secret.json', SCOPES,
                redirect_uri='http://localhost:1',
            )
            auth_url, _ = flow.authorization_url(prompt='consent', access_type='offline')

            print("1. Open this URL in your browser:\n")
            print(auth_url)
            print("\n2. Sign in and grant access.")
            print("3. You'll be redirected to a page that WON'T load — that's expected.")
            print("4. Copy the FULL URL from your browser's address bar and paste it below.")
            print("   (It will look like: http://localhost:1/?code=4/0A...&scope=...)\n")
            redirect_response = input("Paste the full redirect URL here: ").strip()

            flow.fetch_token(authorization_response=redirect_response)
            creds = flow.credentials
        with open(TOKEN_FILE, 'wb') as f:
            pickle.dump(creds, f)

    return creds

creds = authenticate()
youtube = build('youtube', 'v3', credentials=creds)
drive = build('drive', 'v3', credentials=creds)
print("Authenticated successfully!")

## 4. Core Upload Engine

These functions handle the actual streaming upload to YouTube with resumable upload support and progress tracking.

In [ ]:
import io
import time
import requests
import re
from googleapiclient.http import MediaIoBaseUpload, MediaFileUpload
from googleapiclient.errors import HttpError

CHUNK_SIZE = 10 * 1024 * 1024  # 10 MB chunks for resumable upload
MAX_RETRIES = 5

VALID_PRIVACY = ['private', 'unlisted', 'public']
YOUTUBE_CATEGORIES = {
    '1': 'Film & Animation', '2': 'Autos & Vehicles', '10': 'Music',
    '15': 'Pets & Animals', '17': 'Sports', '18': 'Short Movies',
    '19': 'Travel & Events', '20': 'Gaming', '21': 'Videoblogging',
    '22': 'People & Blogs', '23': 'Comedy', '24': 'Entertainment',
    '25': 'News & Politics', '26': 'Howto & Style', '27': 'Education',
    '28': 'Science & Technology', '29': 'Nonprofits & Activism',
}


def upload_to_youtube(media_body, title, description='', tags=None,
                      category_id='22', privacy='private'):
    """Upload video to YouTube with resumable upload and retry logic."""
    if privacy not in VALID_PRIVACY:
        raise ValueError(f"privacy must be one of {VALID_PRIVACY}")

    body = {
        'snippet': {
            'title': title,
            'description': description,
            'tags': tags or [],
            'categoryId': category_id,
        },
        'status': {
            'privacyStatus': privacy,
            'selfDeclaredMadeForKids': False,
        },
    }

    request = youtube.videos().insert(
        part='snippet,status',
        body=body,
        media_body=media_body,
    )

    response = None
    retry_count = 0
    print(f"Uploading: {title}")

    while response is None:
        try:
            status, response = request.next_chunk()
            if status:
                pct = int(status.progress() * 100)
                print(f"  Progress: {pct}%", end='\r')
        except HttpError as e:
            if e.resp.status in [500, 502, 503, 504] and retry_count < MAX_RETRIES:
                retry_count += 1
                wait = 2 ** retry_count
                print(f"  Server error, retrying in {wait}s (attempt {retry_count}/{MAX_RETRIES})...")
                time.sleep(wait)
            else:
                raise

    video_id = response['id']
    print(f"\nUpload complete!")
    print(f"  Video ID: {video_id}")
    print(f"  URL: https://www.youtube.com/watch?v={video_id}")
    print(f"  Studio: https://studio.youtube.com/video/{video_id}/edit")
    return response


def extract_drive_file_id(url_or_id):
    """Extract Google Drive file ID from various URL formats or a raw ID."""
    patterns = [
        r'/file/d/([a-zA-Z0-9_-]+)',
        r'id=([a-zA-Z0-9_-]+)',
        r'/open\?id=([a-zA-Z0-9_-]+)',
    ]
    for pattern in patterns:
        match = re.search(pattern, url_or_id)
        if match:
            return match.group(1)
    if re.match(r'^[a-zA-Z0-9_-]{20,}$', url_or_id):
        return url_or_id
    raise ValueError(f"Could not extract Drive file ID from: {url_or_id}")


print("Upload engine ready.")

## 5a. Upload from Google Drive

Paste a Google Drive **share link** or **file ID**. The video streams from Drive → YouTube without saving locally.

In [ ]:
#@title Google Drive → YouTube Upload
#@markdown ### Video Source
drive_url_or_id = "" #@param {type:"string"}

#@markdown ### YouTube Video Details
title = "My Video" #@param {type:"string"}
description = "" #@param {type:"string"}
tags = "" #@param {type:"string"}
privacy = "private" #@param ["private", "unlisted", "public"]
category_id = "22" #@param {type:"string"}

import tempfile

if not drive_url_or_id.strip():
    print("Please enter a Google Drive URL or file ID above.")
else:
    file_id = extract_drive_file_id(drive_url_or_id.strip())
    print(f"Drive file ID: {file_id}")

    file_meta = drive.files().get(fileId=file_id, fields='name,size,mimeType').execute()
    print(f"File: {file_meta['name']}")
    print(f"Size: {int(file_meta.get('size', 0)) / (1024*1024):.1f} MB")
    print(f"Type: {file_meta.get('mimeType', 'unknown')}")

    if not title or title == "My Video":
        name_without_ext = os.path.splitext(file_meta['name'])[0]
        title = name_without_ext

    # Stream from Drive to a temp file, then upload to YouTube.
    # Colab VMs have plenty of /tmp space; this avoids holding the
    # entire file in RAM for large videos.
    print("\nStreaming from Google Drive...")
    tmp_path = os.path.join(tempfile.gettempdir(), file_meta['name'])
    request_dl = drive.files().get_media(fileId=file_id)

    from googleapiclient.http import MediaIoBaseDownload
    with open(tmp_path, 'wb') as f:
        downloader = MediaIoBaseDownload(f, request_dl, chunksize=CHUNK_SIZE)
        done = False
        while not done:
            status, done = downloader.next_chunk()
            if status:
                print(f"  Drive download: {int(status.progress() * 100)}%", end='\r')
    print(f"  Drive download: 100% — streamed to Colab VM")

    print("\nUploading to YouTube...")
    media = MediaFileUpload(
        tmp_path,
        mimetype=file_meta.get('mimeType', 'video/mp4'),
        resumable=True,
        chunksize=CHUNK_SIZE,
    )

    tag_list = [t.strip() for t in tags.split(',') if t.strip()] if tags else []
    result = upload_to_youtube(media, title, description, tag_list, category_id, privacy)

    os.remove(tmp_path)
    print("Temp file cleaned up.")

## 5b. Upload from Direct URL

Paste any **direct video URL** (must be a direct link to a video file, not a webpage). The video streams from the URL → YouTube.

In [ ]:
#@title URL → YouTube Upload
#@markdown ### Video Source
video_url = "" #@param {type:"string"}

#@markdown ### YouTube Video Details
title = "My Video" #@param {type:"string"}
description = "" #@param {type:"string"}
tags = "" #@param {type:"string"}
privacy = "private" #@param ["private", "unlisted", "public"]
category_id = "22" #@param {type:"string"}

import tempfile

if not video_url.strip():
    print("Please enter a video URL above.")
else:
    url = video_url.strip()
    print(f"Fetching: {url}")

    # Stream the video to a temp file on the Colab VM
    resp = requests.get(url, stream=True, timeout=30)
    resp.raise_for_status()

    content_type = resp.headers.get('Content-Type', 'video/mp4')
    content_length = resp.headers.get('Content-Length')
    if content_length:
        print(f"Size: {int(content_length) / (1024*1024):.1f} MB")
    print(f"Type: {content_type}")

    # Determine filename from URL or Content-Disposition
    cd = resp.headers.get('Content-Disposition', '')
    if 'filename=' in cd:
        fname = cd.split('filename=')[-1].strip('"\' ')
    else:
        from urllib.parse import urlparse
        fname = os.path.basename(urlparse(url).path) or 'video.mp4'

    tmp_path = os.path.join(tempfile.gettempdir(), fname)

    print("\nStreaming from URL...")
    downloaded = 0
    with open(tmp_path, 'wb') as f:
        for chunk in resp.iter_content(chunk_size=CHUNK_SIZE):
            if chunk:
                f.write(chunk)
                downloaded += len(chunk)
                if content_length:
                    pct = int(downloaded / int(content_length) * 100)
                    print(f"  Download: {pct}% ({downloaded/(1024*1024):.1f} MB)", end='\r')
                else:
                    print(f"  Downloaded: {downloaded/(1024*1024):.1f} MB", end='\r')
    print(f"\n  Download complete — streamed to Colab VM")

    if not title or title == "My Video":
        title = os.path.splitext(fname)[0]

    print("\nUploading to YouTube...")
    mime = content_type.split(';')[0].strip()
    if not mime.startswith('video/'):
        mime = 'video/mp4'

    media = MediaFileUpload(
        tmp_path,
        mimetype=mime,
        resumable=True,
        chunksize=CHUNK_SIZE,
    )

    tag_list = [t.strip() for t in tags.split(',') if t.strip()] if tags else []
    result = upload_to_youtube(media, title, description, tag_list, category_id, privacy)

    os.remove(tmp_path)
    print("Temp file cleaned up.")

## 6. Batch Upload from Google Drive Folder

Upload **all videos** from a Google Drive folder at once.

In [ ]:
#@title Batch Upload — Drive Folder → YouTube
#@markdown ### Drive Folder
drive_folder_url_or_id = "" #@param {type:"string"}

#@markdown ### YouTube Defaults
default_privacy = "private" #@param ["private", "unlisted", "public"]
default_category_id = "22" #@param {type:"string"}
default_description = "" #@param {type:"string"}

import tempfile
from googleapiclient.http import MediaIoBaseDownload

if not drive_folder_url_or_id.strip():
    print("Please enter a Google Drive folder URL or ID above.")
else:
    folder_id = extract_drive_file_id(drive_folder_url_or_id.strip())
    print(f"Folder ID: {folder_id}")

    query = f"'{folder_id}' in parents and mimeType contains 'video/' and trashed = false"
    results = drive.files().list(
        q=query,
        fields='files(id,name,size,mimeType)',
        orderBy='name',
        pageSize=100,
    ).execute()
    video_files = results.get('files', [])

    if not video_files:
        print("No video files found in this folder.")
    else:
        print(f"Found {len(video_files)} video(s):")
        for i, vf in enumerate(video_files, 1):
            size_mb = int(vf.get('size', 0)) / (1024*1024)
            print(f"  {i}. {vf['name']} ({size_mb:.1f} MB)")

        print(f"\nStarting batch upload...\n")
        for i, vf in enumerate(video_files, 1):
            print(f"{'='*60}")
            print(f"[{i}/{len(video_files)}] {vf['name']}")
            print(f"{'='*60}")

            try:
                tmp_path = os.path.join(tempfile.gettempdir(), vf['name'])
                request_dl = drive.files().get_media(fileId=vf['id'])
                with open(tmp_path, 'wb') as f:
                    downloader = MediaIoBaseDownload(f, request_dl, chunksize=CHUNK_SIZE)
                    done = False
                    while not done:
                        status, done = downloader.next_chunk()
                        if status:
                            print(f"  Drive download: {int(status.progress() * 100)}%", end='\r')
                print(f"  Drive download: 100%")

                vid_title = os.path.splitext(vf['name'])[0]
                media = MediaFileUpload(
                    tmp_path,
                    mimetype=vf.get('mimeType', 'video/mp4'),
                    resumable=True,
                    chunksize=CHUNK_SIZE,
                )
                upload_to_youtube(media, vid_title, default_description, [], default_category_id, default_privacy)
                os.remove(tmp_path)

            except Exception as e:
                print(f"  FAILED: {e}")
                if os.path.exists(tmp_path):
                    os.remove(tmp_path)
                continue

        print(f"\n{'='*60}")
        print(f"Batch upload complete!")
        print(f"{'='*60}")

## 7. Check Upload Quota

YouTube API has a daily quota (default 10,000 units). Each video upload costs **1,600 units**, so you get roughly **6 uploads/day** with a fresh project. You can request a quota increase from Google Cloud Console.

In [ ]:
print("YouTube Data API v3 Quota Info")
print("=" * 40)
print("Default daily quota:    10,000 units")
print("Cost per video upload:  1,600 units")
print("Uploads per day:        ~6 videos")
print()
print("To request higher quota:")
print("  1. Go to Google Cloud Console → APIs & Services → YouTube Data API v3")
print("  2. Click 'Quotas' tab")
print("  3. Click the pencil icon next to 'Queries per day'")
print("  4. Click 'Apply for higher quota'")
print("  5. Fill out the form (may take a few days for approval)")